In [ ]:
#!/usr/bin/env python3
"""诊断 tokenizer.model 文件问题"""
import os
import hashlib

model_path = "/nfs2/zdy_download/internlm2_5-7b-chat/tokenizer.model"

# 1. 文件基本信息
print("=" * 60)
print("1. 文件基本信息")
print("=" * 60)
stat = os.stat(model_path)
print(f"路径: {model_path}")
print(f"大小: {stat.st_size} bytes ({stat.st_size/1024/1024:.2f} MB)")
print(f"修改时间: {os.path.getmtime(model_path)}")

# 2. MD5 校验
print()
print("=" * 60)
print("2. MD5 校验")
print("=" * 60)
with open(model_path, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()
print(f"MD5: {md5}")
print(f"(请对比 HuggingFace 上的正确 MD5)")

# 3. 检查 sentencepiece 版本
print()
print("=" * 60)
print("3. sentencepiece 版本检查")
print("=" * 60)
import sentencepiece as spm
print(f"sentencepiece version: {spm.__version__}")

# 4. 尝试加载
print()
print("=" * 60)
print("4. 尝试加载")
print("=" * 60)
try:
    sp = spm.SentencePieceProcessor()
    sp.Load(model_path)
    print(f"SUCCESS: vocab size = {sp.GetPieceSize()}")
except Exception as e:
    print(f"FAILED: {e}")

# 5. 用 protobuf 解析，找到包含 null 的 piece
print()
print("=" * 60)
print("5. 深入分析：哪些 token 包含 null 字符")
print("=" * 60)
try:
    import sentencepiece_model_pb2 as sp_pb2
    model = sp_pb2.ModelProto()
    with open(model_path, "rb") as f:
        model.ParseFromString(f.read())
    
    null_pieces = []
    for i, piece in enumerate(model.pieces):
        if "\x00" in piece.piece:
            null_pieces.append((i, piece.piece))
    
    if null_pieces:
        print(f"发现 {len(null_pieces)} 个包含 null 字符的 token:")
        for idx, piece_str in null_pieces[:10]:
            print(f"  [{idx}] repr={repr(piece_str)} hex={piece_str.encode().hex()}")
    else:
        print("没有发现包含 null 字符的 token (文件可能没问题)")
except ImportError:
    print("sentencepiece_model_pb2 未安装，跳过此检查")
    print("安装方法: pip install sentencepiece-model")
except Exception as e:
    print(f"protobuf 解析失败: {e}")



In [ ]:
import requests

proxies = {
    "http": "socks5://127.0.0.1:1824",
    "https": "socks5://127.0.0.1:1824",
}

resp = requests.get(
    "https://www.imdb.com/list/ls4154830192/",
    proxies=proxies,
    verify=False,   # 仅测试，生产环境不要这么干
    timeout=15,
)
